In [4]:
# %% [markdown]
# BEACON — Complete Net + Mem EDA Pipeline
#
# Open this file in VS Code (Python + Jupyter extensions installed).
# Each `# %%` block is one runnable cell — run top to bottom with Shift+Enter.
#
# Assumes folder structure:
#   D:\Malware Dataset\NetCSVs\<Category>\*.csv
#   D:\Malware Dataset\MemoryCSVs\<Category>\*.csv
# where <Category> is one of: Backdoor, Benign, Exploit, HackTool, Hoax,
# Rootkit, Trojan, Virus, Worm

# %%
# ============================================================
# STEP 1: Imports + configuration — EDIT THESE PATHS
# ============================================================
import glob
import os
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# EDIT to match your actual drive/folder locations
NET_ROOT = r"D:\Malware Dataset\NetCSVs"
MEM_ROOT = r"D:\Malware Dataset\MemoryCSVs"

CATEGORIES = ["Spyware"]

OUTPUT_DIR = r"D:\Malware Dataset\processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# %%
# ============================================================
# STEP 2: Discover all files per category (sanity check first)
# ============================================================
net_files_by_cat = {}
mem_files_by_cat = {}

print("Network files:")
for cat in CATEGORIES:
    files = glob.glob(os.path.join(NET_ROOT, cat, "*.csv"))
    net_files_by_cat[cat] = files
    print(f"  {cat}: {len(files)} files")

print("\nMemory files:")
for cat in CATEGORIES:
    files = glob.glob(os.path.join(MEM_ROOT, cat, "*.csv"))
    mem_files_by_cat[cat] = files
    print(f"  {cat}: {len(files)} files")

# Flag any empty categories before wasting time downstream
empty_net = [c for c, f in net_files_by_cat.items() if len(f) == 0]
empty_mem = [c for c, f in mem_files_by_cat.items() if len(f) == 0]
if empty_net:
    print(f"\nWARNING: no network files found for: {empty_net}")
if empty_mem:
    print(f"WARNING: no memory files found for: {empty_mem}")

# %%
# ============================================================
# STEP 3: Check whether files already contain a 'label' column
# ============================================================
# Individual per-sample CSVs (like your 428_3.csv) often do NOT carry
# a label column — the folder name IS the label. Check one sample file
# per category to confirm, for both sources.
def check_label_column(files_by_cat: dict, source_name: str):
    print(f"\n--- {source_name} ---")
    for cat, files in files_by_cat.items():
        if not files:
            continue
        cols = pl.scan_csv(files[0], ignore_errors=True).collect_schema().names()
        has_label = "label" in cols
        print(f"  {cat}: {'has label column' if has_label else 'NO label column — will tag from folder name'}")

check_label_column(net_files_by_cat, "Network")
check_label_column(mem_files_by_cat, "Memory")

# %%
# ============================================================
# STEP 4: Load helper — adds label + sample_id from filename/folder
# ============================================================
# sample_id is derived from the filename (without extension), so a file
# like "428_3.csv" becomes sample_id "428_3". This is what lets you join
# network and memory records for the same sample later.
def load_with_metadata(filepath: str, category: str) -> pl.LazyFrame:
    sample_id = os.path.splitext(os.path.basename(filepath))[0]
    lf = pl.scan_csv(filepath, ignore_errors=True)
    cols = lf.collect_schema().names()

    exprs = [pl.lit(sample_id).alias("sample_id")]
    if "label" not in cols:
        exprs.append(pl.lit(category).alias("label"))

    return lf.with_columns(exprs)

# %%
# ============================================================
# STEP 5: Load + concatenate ALL network files across all categories
# ============================================================
net_lazy_frames = []
for cat, files in net_files_by_cat.items():
    for f in files:
        net_lazy_frames.append(load_with_metadata(f, cat))

net_df = pl.concat(net_lazy_frames, how="diagonal_relaxed")  # tolerates minor schema drift across files
print("Total network columns:", len(net_df.collect_schema().names()))
print("Total network rows:", net_df.select(pl.len()).collect()[0, 0])

# %%
# ============================================================
# STEP 6: Load + concatenate ALL memory files across all categories
# ============================================================
mem_lazy_frames = []
for cat, files in mem_files_by_cat.items():
    for f in files:
        mem_lazy_frames.append(load_with_metadata(f, cat))

mem_df = pl.concat(mem_lazy_frames, how="diagonal_relaxed")
print("Total memory columns:", len(mem_df.collect_schema().names()))
print("Total memory rows:", mem_df.select(pl.len()).collect()[0, 0])

# %%
# ============================================================
# STEP 7: Label distribution — check class balance for BOTH sources
# ============================================================
net_label_dist = net_df.group_by("label").agg(pl.len().alias("count")).collect().sort("count", descending=True)
print("Network label distribution:")
print(net_label_dist)

mem_label_dist = mem_df.group_by("label").agg(pl.len().alias("count")).collect().sort("count", descending=True)
print("\nMemory label distribution:")
print(mem_label_dist)

net_label_dist.to_pandas().plot(kind="bar", x="label", y="count", legend=False, title="Network — class balance")
plt.tight_layout()
plt.show()

mem_label_dist.to_pandas().plot(kind="bar", x="label", y="count", legend=False, title="Memory — class balance")
plt.tight_layout()
plt.show()

# %%
# ============================================================
# STEP 8: Missing values check (both sources)
# ============================================================
def null_report(df: pl.LazyFrame, name: str):
    cols = df.collect_schema().names()
    nulls = df.select([pl.col(c).is_null().sum().alias(c) for c in cols]).collect()
    nulls_pd = nulls.to_pandas().T
    nulls_pd.columns = ["null_count"]
    found = nulls_pd[nulls_pd["null_count"] > 0].sort_values("null_count", ascending=False)
    print(f"\n{name} — columns with missing values:")
    print(found if len(found) else "  none")
    return found

net_nulls = null_report(net_df, "Network")
mem_nulls = null_report(mem_df, "Memory")

# %%
# ============================================================
# STEP 9: Known data quality fix — handshake_state / delta_start
# ============================================================
# Some rows contain the literal text "not a complete handshake" in what
# should be a numeric column. ignore_errors=True already nulled these out
# during load — this step makes that explicit as its own flag instead of
# silently treating it as missing data.
if "handshake_state" in net_df.collect_schema().names():
    net_df = net_df.with_columns(
        pl.col("handshake_state").is_null().alias("incomplete_handshake_flag")
    )
    incomplete_count = net_df.select(pl.col("incomplete_handshake_flag").sum()).collect()[0, 0]
    print(f"Flows with incomplete handshake: {incomplete_count}")

# %%
# ============================================================
# STEP 10: Duplicate check
# ============================================================
net_dupes = net_df.collect().is_duplicated().sum()
print(f"Duplicate rows in network data: {net_dupes}")

# %%
# ============================================================
# STEP 11: Summary statistics on key network features
# ============================================================
key_cols = ["duration", "packets_count", "total_payload_bytes", "bytes_rate", "packets_rate"]
available_key_cols = [c for c in key_cols if c in net_df.collect_schema().names()]
print(net_df.select(available_key_cols).collect().describe())

# %%
# ============================================================
# STEP 12: Sample down for visuals — NEVER plot the full dataset at scale
# ============================================================
total_rows = net_df.select(pl.len()).collect()[0, 0]
sample_size = min(200_000, total_rows)
sample = net_df.collect().sample(n=sample_size, seed=42).to_pandas()
print(f"Sampled {sample_size} rows for visualization")

# %%
# ============================================================
# STEP 13: Feature distributions (histogram grid)
# ============================================================
numeric_cols = sample.select_dtypes(include="number").columns[:9]
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.flat, numeric_cols):
    sample[col].hist(bins=40, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

# %%
# ============================================================
# STEP 14: Correlation heatmap (sampled, numeric columns)
# ============================================================
numeric_sample = sample.select_dtypes(include="number")
numeric_sample = numeric_sample.loc[:, numeric_sample.std(numeric_only=True) > 0]  # drop zero-variance cols
corr = numeric_sample.iloc[:, :40].corr()  # first 40 for readability

plt.figure(figsize=(16, 13))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Feature correlation (first 40 numeric columns, sampled)")
plt.tight_layout()
plt.show()

# %%
# ============================================================
# STEP 15: Feature vs label relationships
# ============================================================
for col in available_key_cols:
    plt.figure()
    sns.boxplot(x="label", y=col, data=sample)
    plt.title(f"{col} by label")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# %%
# ============================================================
# STEP 16: Merge network + memory features on sample_id
# ============================================================
merged = net_df.join(mem_df, on="sample_id", how="inner", suffix="_mem")
merged_rows = merged.select(pl.len()).collect()[0, 0]
print(f"Merged rows (matched sample_id in both sources): {merged_rows}")
print(f"Network-only rows: {total_rows} | Match rate: {merged_rows / total_rows * 100:.1f}%")

# A low match rate here means sample_id naming differs between your
# NetCSVs and MemoryCSVs files — check actual filenames if this is low.

# %%
# ============================================================
# STEP 17: Save the cleaned, merged feature table
# ============================================================
# sink_parquet streams the result to disk without holding it fully in RAM —
# important given your dataset's total size.
output_path = os.path.join(OUTPUT_DIR, "beacon_clean_features.parquet")
merged.sink_parquet(output_path)
print(f"Saved: {output_path}")

Network files:
  Spyware: 0 files

Memory files:
  Spyware: 3 files


--- Network ---

--- Memory ---
  Spyware: NO label column — will tag from folder name


ValueError: cannot concat empty list